# Testing Notebook

## Setup required here

In [ ]:
# !pip install pandas
# !pip install langchain_openai
# !pip install langchain
# !pip install python-dotenv


In [ ]:
import sys
import os

# Go up one level to the project root, then into the utils folder
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
import pandas as pd

# Display BBC News Data
bbc_news_df = pd.read_csv('../csv/bbc-news-data.csv', sep='\t')
bbc_news_df.head(1)

In [ ]:
# Display Taylor Swift Data
taylor_swift_df = pd.read_csv('../csv/TaylorSwift13.csv', sep=',')
taylor_swift_df.head(1)

In [ ]:
taylor_swift_df.columns

In [ ]:
# ========================================
# PHASE 1: DATA SAMPLING (Data Sieve)
# ========================================

def prepare_style_samples(bbc_path, ts_path):
    """
    Prepare style samples from BBC and Taylor Swift datasets.
    
    BBC: Group by category and pick 2 samples from each for diversity
    Taylor Swift: Sort by engagement and pick top 20 tweets
    """
    # Load Datasets
    bbc_df = pd.read_csv(bbc_path, sep='\t')
    ts_df = pd.read_csv(ts_path, sep=',')
    
    print("=== BBC News Dataset ===")
    print(f"Total articles: {len(bbc_df)}")
    print(f"Categories: {bbc_df['category'].unique()}")
    
    print("\n=== Taylor Swift Dataset ===")
    print(f"Total tweets: {len(ts_df)}")
    print(f"Columns: {ts_df.columns.tolist()}")
    
    # 1. BBC Sampling
    bbc_samples = bbc_df.groupby('category').head(2)
    bbc_text = "\n---\n".join(
        bbc_samples['title'] + ": " + bbc_samples['content'].str[:500]
    )
    
    # 2. Taylor Swift Sampling
    engagement_col = None
    for col in ['likeCount', 'likes', 'favorite_count', 'favorites']:
        if col in ts_df.columns:
            engagement_col = col
            break
    
    if engagement_col:
        ts_samples = ts_df.sort_values(by=engagement_col, ascending=False).head(20)
        print(f"\nUsing '{engagement_col}' for Taylor Swift sampling")
    else:
        ts_samples = ts_df.head(20)
        print("\nNo engagement column found, using first 20 tweets")
    
    ts_text = "\n---\n".join(ts_samples['content'].astype(str))
    
    return bbc_text, ts_text

bbc_corpus, ts_corpus = prepare_style_samples('../csv/bbc-news-data.csv', '../csv/TaylorSwift13.csv')

In [ ]:
# ========================================
# PHASE 2: LANGCHAIN SETUP
# ========================================

from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()

model = os.getenv("LLM_MODEL_NAME", "gpt-5-nano")

llm = ChatOpenAI(
    model=model,
    temperature=0.7
)

print(f"✅ LangChain initialized with '{model}'")

In [ ]:
# ========================================
# LOGGING SETUP
# ========================================

import sys
sys.path.append('utils')

from utils.logging_utils import init_logs, logged_invoke, get_logs_dataframe, get_summary, list_log_files

RUN_ID = None

init_logs(RUN_ID)

print("✅ Logging initialized")
print(f"Log files available: {list_log_files()}")

In [ ]:
# ========================================
# PHASE 3: CHARACTERIZATION AGENT
# ========================================

def extract_style_guide(corpus, source_type, use_logging=True):

    prompt = f"""I am providing you with a corpus of {source_type} texts.
Your task is to act as a Linguistic Pattern Analyst.

Analyze and output JSON with:

Tone
Structure
Punctuation Quirk
Vocabulary Level
Formatting
Key Phrases

=== CORPUS ===
{corpus[:3000]}
=== END CORPUS ===

JSON Output:"""

    if use_logging:

        task_name = f"{source_type}_Style_Extraction"

        response_content = logged_invoke(
            llm,
            prompt,
            task_name=task_name,
            run_id=RUN_ID
        )

        return response_content

    else:

        response = llm.invoke(prompt)

        return response.content


print("Extracting BBC style guide...")

bbc_style_guide = extract_style_guide(
    bbc_corpus,
    "BBC News Articles"
)

print(bbc_style_guide)

In [ ]:
print("Extracting Taylor Swift style guide...")

ts_style_guide = extract_style_guide(
    ts_corpus,
    "Taylor Swift Tweets"
)

print(ts_style_guide)

In [ ]:
df_logs = get_logs_dataframe(RUN_ID)
display(df_logs)

In [ ]:
summary = get_summary(RUN_ID)
print(summary)

In [ ]:
style_guides = {
    "bbc": bbc_style_guide,
    "taylor_swift": ts_style_guide
}

import json

with open('../artifact/style_guides.json', 'w') as f:
    json.dump(style_guides, f, indent=2)

print("Saved successfully")